#Downloading libs

In [ ]:
!pip install arxiv requests tqdm

In [ ]:
!pip install -q tiktoken

In [ ]:
!pip install -q sentence-transformers qdrant-client

In [ ]:
!pip install -q rank_bm25 transformers accelerate

In [ ]:
!pip install -q pymupdf4llm pymupdf tqdm

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/ProjectsDataCloud/arxiv-rag")   # غيّره لمكان مشروعك
# لو شغال على /content مباشرة: BASE = Path("/content/arxiv-rag")

Mounted at /content/drive


# Importing Libs

In [ ]:
import json
import time
from pathlib import Path

import arxiv
import requests
from tqdm import tqdm
from collections import Counter

# Downloading Data

In [ ]:
"""
Download arXiv papers + metadata.
Resumable: re-running skips PDFs that already exist.
"""

# ----------Config------------
QUERY= 'abs:"retrieval augmented generation" AND cat:cs.CL'
MAX_RESULTS = 300
PDF_DIR    = BASE / "data/raw_pdfs"
META_PATH  = BASE / "data/metadata/papers.jsonl"
PARSED_DIR = BASE / "data/parsed"
REQUEST_DELAY = 3
DROP_REFERENCES = True
#-----------------------------

PDF_DIR.mkdir(parents=True, exist_ok=True)
META_PATH.parent.mkdir(parents=True, exist_ok= True)
PARSED_DIR.mkdir(parents=True, exist_ok=True)

def load_existing_ids() ->set[str]:
  """ To get meta data  """
  if not META_PATH.exists():
    return set()
  with META_PATH.open(encoding="utf-8") as f:
    return {json.loads(line)["arxiv_id"] for line in f if line.strip()}

def download_pdf(url: str, dest: Path) ->bool:
  tmp = dest.with_suffix(".part")
  try:
        r = requests.get(url, timeout=60, stream=True)
        r.raise_for_status()
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 15):
                f.write(chunk)
        tmp.rename(dest)
        return True
  except Exception as e:
        print(f"  ! failed {url}: {e}")
        tmp.unlink(missing_ok=True)
        return False


In [ ]:
client = arxiv.Client(page_size= 100, delay_seconds=REQUEST_DELAY,num_retries=5)
search = arxiv.Search(
    query=QUERY,
    max_results=MAX_RESULTS,
    sort_by=arxiv.SortCriterion.Relevance
)

done_ids = load_existing_ids()
print(f"Already have {len(done_ids)} papers. Fetching up to {MAX_RESULTS}")

with META_PATH.open("a",encoding="utf-8") as f:
  for result in tqdm(client.results(search), total=MAX_RESULTS):
            arxiv_id = result.get_short_id()          # مثال: 2005.11401v4
            base_id = arxiv_id.split("v")[0]          # 2005.11401
            if base_id in done_ids:
                continue

            pdf_path = PDF_DIR / f"{base_id.replace('/', '_')}.pdf"
            if not pdf_path.exists():
                ok = download_pdf(result.pdf_url, pdf_path)
                time.sleep(REQUEST_DELAY)
                if not ok:
                    continue

            record = {
                "arxiv_id": base_id,
                "title": result.title.strip().replace("\n", " "),
                "authors": [a.name for a in result.authors],
                "abstract": result.summary.strip().replace("\n", " "),
                "published": result.published.isoformat(),
                "updated": result.updated.isoformat(),
                "primary_category": result.primary_category,
                "categories": result.categories,
                "pdf_url": result.pdf_url,
                "pdf_path": str(pdf_path),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            done_ids.add(base_id)

  print(f"\nDone. Total papers: {len(done_ids)}")

Already have 297 papers. Fetching up to 300


 34%|███▎      | 101/300 [00:03<00:07, 27.01it/s]

  ! failed https://arxiv.org/pdf/2608.12129v2: 404 Client Error: Not Found for url: https://arxiv.org/pdf/2608.12129v2


 67%|██████▋   | 201/300 [00:07<00:03, 32.69it/s]

  ! failed https://arxiv.org/pdf/2504.02894v3: 404 Client Error: Not Found for url: https://arxiv.org/pdf/2504.02894v3


 79%|███████▉  | 238/300 [00:10<00:02, 22.56it/s]

  ! failed https://arxiv.org/pdf/2408.13366v2: 404 Client Error: Not Found for url: https://arxiv.org/pdf/2408.13366v2


100%|██████████| 300/300 [00:13<00:00, 21.92it/s]


Done. Total papers: 297


In [ ]:
# Checking Data


records = [json.loads(l) for l in META_PATH.open(encoding="utf-8") if l.strip()]
print(f"Papers in metadata: {len(records)}")

missing, tiny = [], []
for r in records:
    p = Path(r["pdf_path"])
    if not p.exists():
        missing.append(p)
    elif p.stat().st_size < 50_000:        # أقل من 50KB مشبوه
        tiny.append((p, p.stat().st_size))

print(f"Missing PDFs: {len(missing)}")
print(f"Suspiciously small PDFs: {len(tiny)}")

years = Counter(r["published"][:4] for r in records)
print("Papers per year:", dict(sorted(years.items())))

sizes = [Path(r["pdf_path"]).stat().st_size / 1e6 for r in records if Path(r["pdf_path"]).exists()]
print(f"Total size: {sum(sizes):.0f} MB | avg: {sum(sizes)/len(sizes):.2f} MB")

Papers in metadata: 297
Missing PDFs: 0
Suspiciously small PDFs: 0
Papers per year: {'2021': 1, '2022': 2, '2023': 11, '2024': 92, '2025': 120, '2026': 71}
Total size: 585 MB | avg: 1.97 MB


# Parsing& Cleaning

In [ ]:
import json, re
from collections import Counter
from tqdm.auto import tqdm
import pymupdf4llm

In [ ]:
def parse_pdf(pdf_path: Path) -> list[dict]:
  """
  retrun list: pages = {'page': int, 'text': markdown}
  """
  chunks = pymupdf4llm.to_markdown(str(pdf_path),page_chunks=True, show_progress=False)
  return [{'page':i+1, 'text':c['text']} for i, c in enumerate(chunks)]


def find_repeated_lines(pages: list[dict], min_ratio=.3, max_len=120) -> set[str]:
  """
  Header/Footer = small lines repeated in more than 30% of pages
  """
  counter = Counter()
  for p in pages:
    lines = {l.strip() for l in p['text'].splitlines() if l.strip()}
    counter.update(l for l in lines if len(l) <=max_len)
  threshold = max(3, int(len(pages))*min_ratio)
  return {l for l, c in counter.items() if c>= threshold}

ARXIV_STAMP = re.compile(r"arXiv:\d{4}\.\d{4,5}v\d+\s+\[[\w\.\-]+\]\s+\d+\s+\w+\s+\d{4}")
PAGE_NUM    = re.compile(r"^\s*(page\s*)?\d{1,4}\s*$", re.I)


def clean_page(text: str, repeated: set[str]) -> str:
    text = ARXIV_STAMP.sub("", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # كلمات مقسومة بـ hyphen
    lines = []
    for l in text.splitlines():
        s = l.strip()
        if s in repeated or PAGE_NUM.match(s):
            continue
        lines.append(l.rstrip())
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)                 # شيل الفراغات الكتيرة
    return text.strip()


REF_HEADING = re.compile(r"^\s*(#{1,4}\s*|\*\*)?\s*(references|bibliography)\s*(\*\*)?\s*$", re.I | re.M)

def cut_references(pages: list[dict]) -> tuple[list[dict], bool]:
    """يقطع من أول عنوان References ويحذف اللي بعده."""
    for i, p in enumerate(pages):
        m = REF_HEADING.search(p["text"])
        if m:
            p["text"] = p["text"][: m.start()].strip()
            return [q for q in pages[: i + 1] if q["text"]], True
    return pages, False

In [ ]:
records = [json.loads(l) for l in META_PATH.open(encoding="utf-8") if l.strip()]
failed = []

for rec in tqdm(records):
  out_path = PARSED_DIR / f"{rec['arxiv_id'].replace('/', '_')}.json"
  if out_path.exists():
    continue
  try:
    pdf_path = PDF_DIR / Path(rec["pdf_path"]).name
    pages = parse_pdf(pdf_path)
    repeated = find_repeated_lines(pages)
    for p in pages:
      p["text"] = clean_page(p["text"], repeated)
    pages = [p for p in pages if p["text"]]

    refs_cut = False
    if DROP_REFERENCES:
      pages, refs_cut = cut_references(pages)

    doc = {
        "arxiv_id": rec["arxiv_id"],
        "title": rec["title"],
        "published": rec["published"],
        "authors": rec["authors"],
        "pdf_url": rec["pdf_url"],
        "references_removed": refs_cut,
        "pages": pages,
    }
    out_path.write_text(json.dumps(doc, ensure_ascii=False), encoding="utf-8")
  except Exception as e:
    failed.append((rec["arxiv_id"], str(e)))

print(f"Parsed: {len(list(PARSED_DIR.glob('*.json')))} | Failed: {len(failed)}")
for f in failed[:10]:
  print(f)

  0%|          | 0/297 [00:00<?, ?it/s]

Parsed: 297 | Failed: 0


In [ ]:
import random, statistics

docs = [json.loads(p.read_text(encoding="utf-8")) for p in PARSED_DIR.glob("*.json")]

chars_per_doc = [sum(len(p["text"]) for p in d["pages"]) for d in docs]
print(f"Docs: {len(docs)}")
print(f"Chars/doc  median: {statistics.median(chars_per_doc):,.0f} | min: {min(chars_per_doc):,} | max: {max(chars_per_doc):,}")
print(f"References removed in: {sum(d['references_removed'] for d in docs)}/{len(docs)} docs")

# مستندات مشبوهة (نص قليل = ممكن PDF مصوّر/scanned أو parsing فاشل)
suspicious = [d["arxiv_id"] for d, c in zip(docs, chars_per_doc) if c < 5000]
print("Suspicious (<5k chars):", suspicious)

# اقرا عينة بعينك
d = random.choice(docs)
print("\n=====", d["title"], "=====")
print(d["pages"][0]["text"][:1500])

Docs: 297
Chars/doc  median: 49,848 | min: 8,065 | max: 1,398,897
References removed in: 40/297 docs
Suspicious (<5k chars): []

===== ColMate: Contrastive Late Interaction and Masked Text for Multimodal Document Retrieval =====
# **COLMATE : Contrastive Late Interaction and Masked Text for Multimodal Document Retrieval**

**Ahmed Masry**<sup>_♣♡_</sup> **, Megh Thakkar**<sup>_♣_</sup> **, Patrice Bechard**<sup>_♣_</sup> **, Sathwik Tejaswi Madhusudhan**<sup>_♣_</sup> **, Rabiul Awal**<sup>_♣♠♢_</sup> **, Shambhavi Mishra**<sup>_♣△_</sup> **, Akshay Kalkunte Suresh**<sup>_♣_</sup> **, Srivatsava Daruru**<sup>_♣_</sup> **, Enamul Hoque**<sup>_♡_</sup> **, Spandana Gella**<sup>_♣_</sup> **, Torsten Scholak**<sup>_♣_</sup> **, Sai Rajeswar**<sup>_♣♠♢_*</sup>

> _♣_ ServiceNow, _♡_ York University, _♠_ MILA - Quebec AI Institute,

> _♢_ Université de Montréal, _△_ École de technologie supérieure

## **Abstract**

Retrieval-augmented generation has proven practical when models require speci

# Chunking

In [ ]:
import json, re, hashlib
from pathlib import Path
from tqdm.auto import tqdm
import tiktoken

CHUNKS_DIR = BASE / "data/chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
_tok.model_max_length = 10**7          # يمنع تحذير الطول

def n_tokens(text: str) -> int:
    return len(_tok.encode(text, add_special_tokens=False))

CHUNK_TOKENS     = 450      # هامش أمان تحت 512
OVERLAP_TOKENS   = 60
MIN_CHUNK_TOKENS = 40
PARENT_TOKENS    = 1500

def make_id(*parts):
  return hashlib.md5("|".join(map(str,parts)).encode()).hexdigest()[:12]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

## Fixed-Size chunking


In [ ]:
def split_sentences(text: str) -> list[str]:
  parts = re.split(r"(?<=[.!?])\s+|\n{2,}", text)
  return [p.strip() for p in parts if p and p.strip()]

def fixed_size_chunks(pages: list[dict], max_tokens=CHUNK_TOKENS, overlap= OVERLAP_TOKENS) -> list[dict]:
  units= []  #(sentences, page)
  for p in pages:
    for s in split_sentences(p['text']):
      units.append((s, p['page']))

  chunks, cur, cur_tok = [],[],0
  """
  chunks -> For final result
  cur -> For collecting sentences for current chunk [(sen, page)]
  cur_tok -> For counting tokens in current chunk
  """
  for sent, page in units:
    t = n_tokens(sent)  # if sentence is too large, split it into chunks.
    if t > max_tokens:           # جملة/جدول ضخم: اقطعه بالقوة
      offs = _tok(sent, add_special_tokens=False,
                  return_offsets_mapping=True)["offset_mapping"]
      step = max_tokens - overlap
      for i in range(0, len(offs), step):
          j = min(i + max_tokens, len(offs)) - 1
          chunks.append({"text": sent[offs[i][0]:offs[j][1]], "page": page})
      continue
    if cur_tok +t > max_tokens and cur:
      chunks.append({'text':" ".join(s for s,_ in cur),'page':cur[0][1]})
      # begin in next itr at Overlap
      keep, kt =[],0
      for s, pg in reversed(cur):
        st =n_tokens(s)
        if kt + st > overlap:
          break
        keep.insert(0,(s,pg)); kt+= st
      cur, cur_tok = keep, kt
    cur.append((sent, page)); cur_tok += t
  if cur:
    chunks.append({'text':" ".join(s for s,_ in cur),'page':cur[0][1]})
  return chunks

## Structure-Aware chunking

In [ ]:
from sys import prefix
HEADER_RE = re.compile(r"^(#{1,4})\s+(.+?)\s*$", re.M)

def split_by_headers(pages: list[dict]):
    sections, stack = [], {}
    cur_lines, cur_page, cur_path = [], 1, ""

    def flush():
        txt = "\n".join(cur_lines).strip()
        if txt:
            sections.append({"path": cur_path, "text": txt, "page": cur_page})

    for p in pages:
        for line in p["text"].splitlines():
            m = HEADER_RE.match(line)
            if m:
                flush()
                level, title = len(m.group(1)), m.group(2).strip("* ")
                stack = {l: t for l, t in stack.items() if l < level}
                stack[level] = title
                cur_path = " > ".join(stack[k] for k in sorted(stack))
                cur_lines, cur_page = [], p["page"]
            else:
                if not cur_lines:
                    cur_page = p["page"]
                cur_lines.append(line)

    flush()                 # <-- على مستوى الدالة، مش جوه أي loop
    return sections         # <-- نفس الكلام


def structure_chunks(pages: list[dict], title: str, max_tokens=CHUNK_TOKENS, overlap=OVERLAP_TOKENS) -> list[dict]:
  out =[]
  for sec in split_by_headers(pages):
    sub = fixed_size_chunks([{'page':sec['page'],'text':sec['text']}], max_tokens-30,overlap) # Giving each section to fixed-size-split function
    for s in sub:
      prefix=f"{title}"
      if sec['path']:
        prefix += f'| {sec['path']}'
        out.append({
            'text': f"{prefix}\n{s['text']}",
            'page': s['page'],
            'raw_text':s['text'],
            'section':sec['path']
        })
  return out

In [ ]:
def attach_parents(chunks: list[dict], doc_id: str, parent_tokens=PARENT_TOKENS) :
  """collect chunks together to make parents with PARENT_TOKENS tokens"""
  parent_idx, acc, pid = 0, 0, make_id(doc_id, 'parent', 0)
  """
  parent_idx -> index of current parent chunk
  acc -> number of tokens in current parent
  pid -> id of current parent
  """
  parents = {}
  for c in chunks:
    t = n_tokens(c['text'])
    if acc +t >parent_tokens and acc>0:
      parent_idx += 1
      acc,pid = 0, make_id(doc_id, 'parent', parent_idx)
    c["parent_id"] = pid # Attaching chunks with its parents
    parents.setdefault(pid,[]).append(c.get('raw_text',c['text']))
    acc += t
  parent_texts = {k: '\n'.join(v) for k,v in parents.items()}
  return chunks, parent_texts



def process_doc(doc: dict, strategy: str):
    if strategy == "fixed":
        chunks = fixed_size_chunks(doc["pages"])
        for c in chunks:
            c["section"], c["raw_text"] = "", c["text"]
    else:
        chunks = structure_chunks(doc["pages"], doc["title"])

    chunks = [c for c in chunks if n_tokens(c["raw_text"]) >= MIN_CHUNK_TOKENS]
    chunks, parent_texts = attach_parents(chunks, doc["arxiv_id"])

    for i, c in enumerate(chunks):
        c.update({
            "chunk_id": make_id(doc["arxiv_id"], strategy, i),
            "arxiv_id": doc["arxiv_id"],
            "title": doc["title"],
            "published": doc["published"],
            "pdf_url": doc["pdf_url"],
            "strategy": strategy,
            "n_tokens": n_tokens(c["text"]),
        })
    return chunks, parent_texts


for strategy in ["fixed", "structure"]:
    out_path = CHUNKS_DIR / f"chunks_{strategy}.jsonl"
    parents_path = CHUNKS_DIR / f"parents_{strategy}.json"
    all_parents, total = {}, 0

    with out_path.open("w", encoding="utf-8") as f:
        for p in tqdm(sorted(PARSED_DIR.glob("*.json")), desc=strategy):
            doc = json.loads(p.read_text(encoding="utf-8"))
            chunks, parents = process_doc(doc, strategy)
            all_parents.update(parents)
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
            total += len(chunks)

    parents_path.write_text(json.dumps(all_parents, ensure_ascii=False), encoding="utf-8")
    print(f"{strategy}: {total} chunks, {len(all_parents)} parents")


fixed:   0%|          | 0/297 [00:00<?, ?it/s]

fixed: 12922 chunks, 4138 parents


structure:   0%|          | 0/297 [00:00<?, ?it/s]

structure: 16443 chunks, 4425 parents


In [ ]:
import statistics, random

for strategy in ["fixed", "structure"]:
    rows = [json.loads(l) for l in (CHUNKS_DIR / f"chunks_{strategy}.jsonl").open(encoding="utf-8")]
    toks = [r["n_tokens"] for r in rows]
    print(f"\n=== {strategy} ===")
    print(f"chunks: {len(rows)} | tokens median: {statistics.median(toks):.0f} "
          f"| min: {min(toks)} | max: {max(toks)}")
    print(f"chunks without section: {sum(1 for r in rows if not r['section'])}")

    r = random.choice(rows)
    print(f"\n--- sample (page {r['page']}, section: {r['section']!r}) ---")
    print(r["text"][:900])


=== fixed ===
chunks: 12922 | tokens median: 433 | min: 41 | max: 506
chunks without section: 12922

--- sample (page 8, section: '') ---
- **Named Entity (NE):** Extracting an ordered list of named entities from the retrieved documents and concatenates them. - **BM25:** We select it as one representation of classic retrieval algorithms. - **Contriever:** Our initialization dense retrieve model, fine-tuned on MS MARCO. - **Random:** We random select a sentence from retrieved documents. - **RECOMP[7](extractive):** The SOTA post-retrieve RAG model by extractively summarizing the retrieved documents. - **Upperbound:** Upperbound performance is obtained by traversing all the sentences in the retrieved document w.r.t input query X to achieve the best evaluation score. For QA task, we adopt the following baselines: - **GPT-turbo-3.5:** We directly get the response by inputting query X. - **T5:** We use the off-the-shelf T5-large version. - **RECOMP(abstractive):** The SOTA post-retrieve RA

# Embedding

In [ ]:
import json, numpy as np, torch
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct, Filter, FieldCondition, Range
)


EMB_DIR = BASE / "data/embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

BGE_Q = "Represent this sentence for searching relevant passages: "

MODELS= {
    "bge-small": dict(name="BAAI/bge-small-en-v1.5",q=BGE_Q, p=""),
    "bge-base": dict(name="BAAI/bge-base-en-v1.5",q=BGE_Q, p=""),
    "e5-base": dict(name="intfloat/e5-base-v2",q="query",p="passage: ")
}

cpu


In [ ]:
def load_chunks(strategy):
  path = CHUNKS_DIR / f"chunks_{strategy}.jsonl"
  return [json.loads(l) for l in path.open(encoding="utf-8")]

import random
tok = SentenceTransformer(MODELS["bge-base"]["name"], device="cpu").tokenizer
for strategy in ["fixed", "structure"]:
  rows = load_chunks(strategy)
  sample = random.sample(rows, min(500, len(rows)))
  lens = [len(tok(r["text"],trancation=False)["input_ids"]) for r in sample]
  over = sum(l>512 for l in lens)/ len(lens)
  print(f"{strategy}: chunks={len(rows)} | median={np.median(lens):.0f} | max={max(lens)} | >512: {over:.1%}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

fixed: chunks=12922 | median=433 | max=452 | >512: 0.0%


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (788 > 512). Running this sequence through the model will result in indexing errors


structure: chunks=16443 | median=408 | max=788 | >512: 0.6%


In [ ]:
_models = {}
def get_model(key):
  if key not in _models:
    m = SentenceTransformer(MODELS[key]["name"], device=DEVICE)
    m.max_seq_length = 512
    _models[key] = m
  return _models[key]

def embed_chunks(strategy, model_key, batch_size=64):
  out = EMB_DIR / f"{strategy}__{model_key}.npy"
  rows = load_chunks(strategy)
  if out.exists():
    emb = np.load(out)
    assert len(rows) == len(emb)
    print(f"loaded cached: {out.name}")
    return rows, emb

  cfg = MODELS[model_key]
  texts = [cfg["p"] + r["text"] for r in rows]
  emb = get_model(model_key).encode(
      texts, batch_size=batch_size, normalize_embeddings=True,
      show_progress_bar=True, convert_to_numpy=True).astype("float32")
  np.save(out, emb)
  return rows, emb

In [ ]:
out = EMB_DIR / f"structure__bge-base.npy"
emb = np.load(out)
print(emb.shape)

(16443, 768)


In [ ]:
BASE

PosixPath('/content/drive/MyDrive/ProjectsDataCloud/arxiv-rag')

In [ ]:
client = QdrantClient(path="/content/qdrant_db")      # local، بيتمسح مع الـ runtime، وده مقصود

In [ ]:

def build_collection(strategy, model_key, rows, emb, batch=512):
    name = f"{strategy}__{model_key}"
    if client.collection_exists(name):
        print(f"{name} exists ({client.count(name).count} points)")
        return name

    client.create_collection(
        name,
        vectors_config=VectorParams(size=emb.shape[1], distance=Distance.COSINE),
    )
    for i in tqdm(range(0, len(rows), batch), desc=name):
        pts = []
        for j in range(i, min(i + batch, len(rows))):
            r = rows[j]
            pts.append(PointStruct(
                id=j,                                   # integer ID
                vector=emb[j].tolist(),
                payload={
                    "chunk_id": r["chunk_id"],          # الـ ID الثابت (للـ eval)
                    "arxiv_id": r["arxiv_id"],
                    "title": r["title"],
                    "page": r["page"],
                    "section": r["section"],
                    "parent_id": r["parent_id"],
                    "year": int(r["published"][:4]),    # للـ filtering
                    "text": r["raw_text"],              # من غير الـ prefix، للعرض
                },
            ))
        client.upsert(name, points=pts)
    return name

In [ ]:
MODEL_KEY = "bge-base"

for strategy in ["fixed", "structure"]:
    rows, emb = embed_chunks(strategy, MODEL_KEY)
    print(strategy, emb.shape)
    build_collection(strategy, MODEL_KEY, rows, emb)

loaded cached: fixed__bge-base.npy
fixed (12922, 768)


fixed__bge-base:   0%|          | 0/26 [00:00<?, ?it/s]

loaded cached: structure__bge-base.npy
structure (16443, 768)


structure__bge-base:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
def search(query, strategy="structure", model_key="bge-base", k=5, year_from=None):
    cfg = MODELS[model_key]
    qv = get_model(model_key).encode(cfg["q"] + query, normalize_embeddings=True)
    flt = None
    if year_from:
        flt = Filter(must=[FieldCondition(key="year", range=Range(gte=year_from))])
    res = client.query_points(
        f"{strategy}__{model_key}", query=qv.tolist(),
        limit=k, query_filter=flt, with_payload=True,
    )
    return res.points


def show(points, n_chars=350):
    for i, p in enumerate(points, 1):
        pl = p.payload
        print(f"[{i}] score={p.score:.3f} | {pl['title'][:70]} | p.{pl['page']} | {pl['section'] or '-'}")
        print("    " + pl["text"][:n_chars].replace("\n", " ") + "...\n")


queries = [
    "How does a cross-encoder reranker improve retrieval quality?",
    "What chunk size works best for retrieval augmented generation?",
    "How to evaluate faithfulness of generated answers?",
]
for q in queries:
    print("=" * 100, "\nQ:", q, "\n")
    show(search(q, "structure"))

# Advanced Retrieval (Hybrid + Reranking + Query Transformation + Parent expansion)

## BM25+ DENSE + RRF

In [ ]:
import re, json, time, numpy as np, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

STOP = set("a an the of in on for to and or is are was were be by with as at from that this it its we our".split())

def tokenize(text: str) -> list[str]:
  toks = re.findall(r"[a-z0-9]+(?:[-_.][a-z0-9]+)*", text.lower())
  return [t for t in toks if t not in STOP]

def rrf(rank_lists: list[list[int]], k: int = 60) -> list[tuple[int, float]]:
  """
  rank_list:
    [
      [point1_id, point2_id, point3_id, ...],  # First retrieval method
      [point1_id, point2_id, point3_id, ...],  # Second retrieval method
      ...
    ]
  """
  scores = {}
  for lst in rank_lists:
    for rank, idx in enumerate(lst):
      scores[idx] = scores.get(idx, 0.0)+1.0/ (k+rank+1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


In [ ]:
class HybridRetriever:
  def __init__(self,strategy="structure", model_key="bge-base"):
    self.strategy,self.model_key= strategy, model_key
    self.rows = load_chunks(strategy)
    self.years = np.array([int(r["published"][:4]) for r in self.rows])
    print("Building BM25 ...")
    self.bm25 = BM25Okapi([tokenize(r["text"]) for r in self.rows])
    self.collection = f"{strategy}__{model_key}"
    self.parents = json.loads(
        (CHUNKS_DIR / f"parents_{strategy}.json").read_text(encoding="utf-8")
    )
    self._reranker = None

#-----------Dense----------------
  def dense(self, query, n=50, year_from = None, as_passage= False):
    cfg = MODELS[self.model_key]
    prefix = cfg["p"] if as_passage else cfg["q"]
    qv = get_model(self.model_key).encode(prefix + query, normalize_embeddings=True)
    flt = Filter(must=[FieldCondition(key="year", range=Range(gte=year_from))]) if year_from else None
    res = client.query_points(self.collection, query=qv.tolist(),limit=n, query_filter=flt)
    return [p.id for p in res.points]

#------------Spares---------------
  def spares(self, query, n=50, year_from = None,):
    scores = self.bm25.get_scores(tokenize(query))
    if year_from:
        scores = np.where(self.years >= year_from, scores, -1)
    top = np.argsort(-scores)[:n]
    return [int(i) for i in top if scores[i] > 0]


#------------Reranker--------------
  @property
  def reranker(self):
    if self._reranker is None:
      self._reranker = CrossEncoder("BAAI/bge-reranker-base", device=DEVICE, max_length=512)
    return self._reranker

  def rerank(self, query, ids,top_k=5):
    pairs=[]
    for i in ids:
      r= self.rows[i]
      pairs.append((query, f"{r["title"]} | {r['section']}\n{r['raw_text']}"))
      scores = self.reranker.predict(pairs, batch_size=32, show_progress_bar=False)
      order = np.argsort(-scores)[:top_k]
      return [(ids[j], float(scores[j])) for j in order]

## Query Transformation (Local LLM)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
_llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
_llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype= torch.float16, device_map="auto")

def llm_generate(prompt: str, max_new_tokens=200)->str:
  msg = [
      {
          "role":"user",
          "content":prompt
      }
  ]
  ids = _llm_tok.apply_chat_template(msg, add_generation_prompt=True, return_tensors="pt").to(_llm.device)
  out = _llm.generate(**ids, max_new_tokens=max_new_tokens,do_sample=False)

  return _llm_tok.decode(out[0][ids["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()

def multi_query(q: str, n=3) -> list[str]:
    p = (f"Rewrite the following search question into {n} different search queries "
         f"that use different wording and technical terms. Output one per line, no numbering.\n\n"
         f"Question: {q}")
    lines = [l.strip(" -•\t") for l in llm_generate(p).splitlines() if l.strip()]
    return lines[:n]


def hyde(q: str) -> str:
    p = ("Write a short paragraph (3-4 sentences) in the style of a research paper that "
         f"would answer this question. Do not mention that you are guessing.\n\nQuestion: {q}")
    return llm_generate(p, max_new_tokens=150)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def retrieve(R: HybridRetriever, query,*,hybrid=True,rerank=True,use_multi_query=False,
             use_hyde=False,n_candidates=50,top_k=5,year_from=None,expand_parents=False):
  t0 = time.time()
  queries = [query] + (multi_query(query) if use_multi_query else [])

  rank_lists = []
  for q in queries:
    rank_lists.append(R.dense(q,n_candidates,year_from))
    if hybrid:
      rank_lists.append(R.spares(q,n_candidates,year_from))

  if use_hyde:
    rank_lists.append(R.dense(hyde(query),n_candidates,year_from,as_passage=True))

  fused= [i for i,_ in rrf(rank_lists)][:n_candidates]

  if rerank:
    ranked = R.rerank(query,fused,top_k)
  else:
    ranked = [(i, None) for i in fused[:top_k]]

  results=[]
  for i,s in ranked:
    r = R.rows[i]
    results.append({"idx": i, "chunk_id": r["chunk_id"], "score": s,
                        "title": r["title"], "page": r["page"], "section": r["section"],
                        "text": r["raw_text"], "parent_id": r["parent_id"]})

  if expand_parents:                                  # small-to-big
      seen, expanded = set(), []
      for r in results:
          if r["parent_id"] not in seen:
              seen.add(r["parent_id"])
              expanded.append({**r, "text": R.parents[r["parent_id"]]})
      results = expanded

  return results, time.time() - t0

In [ ]:
R = HybridRetriever("structure", "bge-base")

Building BM25 ...


In [ ]:


def show_results(results, n_chars=250):
    for i, r in enumerate(results, 1):
        sc = f"{r['score']:.3f}" if r["score"] is not None else "-"
        print(f"[{i}] {sc} | {r['title'][:60]} | p.{r['page']} | {r['section'][-40:] or '-'}")
        print("    " + r["text"][:n_chars].replace("\n", " ") + "...")

q = "How does a cross-encoder reranker improve retrieval quality?"

configs = {
    "dense only":        dict(hybrid=False, rerank=False),
    "+ hybrid":          dict(hybrid=True,  rerank=False),
    "+ hybrid + rerank": dict(hybrid=True,  rerank=True),
    "+ multi-query":     dict(hybrid=True,  rerank=True, use_multi_query=True),
    "+ HyDE":            dict(hybrid=True,  rerank=True, use_hyde=True),
}
for name, kw in configs.items():
    res, t = retrieve(R, q, top_k=3, **kw)
    print(f"\n########## {name} ({t:.2f}s) ##########")
    show_results(res)

Building BM25 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


########## dense only (4.12s) ##########
[1] - | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.13 | aluation > 5.3 Results > 5.3.1 Retrieval
    Moving to the next stage of retrieval experiments, we evaluated the effect of adding a cross-encoder re-ranking step on both BM25 and Nomic. Improvements were observed across all metrics, with MRR increasing to 0.803 and MAP increasing to 0.499. BM25...
[2] - | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.6 | 3 Methodology > 3.4 Re-ranking
    To improve retrieval precision, we employed the `BAAI/bge-reranker-v2-m3` crossencoder model [33]. Based on an XLM-RoBERTa architecture with a sequence classification head, the model jointly encodes a query and candidate document to produce a scalar ...
[3] - | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.13 | aluation > 5.3 Results > 5.3.1 Retrieval
    Given the observed importance of the re-ranking step, we further fine-tuned the cross-encode

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


########## + hybrid + rerank (15.92s) ##########
[1] 0.933 | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.13 | aluation > 5.3 Results > 5.3.1 Retrieval
    Moving to the next stage of retrieval experiments, we evaluated the effect of adding a cross-encoder re-ranking step on both BM25 and Nomic. Improvements were observed across all metrics, with MRR increasing to 0.803 and MAP increasing to 0.499. BM25...

########## + multi-query (3.16s) ##########
[1] 0.933 | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.13 | aluation > 5.3 Results > 5.3.1 Retrieval
    Moving to the next stage of retrieval experiments, we evaluated the effect of adding a cross-encoder re-ranking step on both BM25 and Nomic. Improvements were observed across all metrics, with MRR increasing to 0.803 and MAP increasing to 0.499. BM25...

########## + HyDE (4.84s) ##########
[1] 0.933 | Grounding Arabic LLMs in the Doha Historical Dictionary: Ret | p.13 | aluation > 5.3 Results 

In [ ]:
for name in ["dense only", "+ hybrid"]:
    res, t = retrieve(R, "What is HyDE and how does it generate hypothetical documents?",
                      top_k=3, **configs[name])
    print(f"\n### {name}"); show_results(res)


### dense only
[1] - | QuOTE: Question-Oriented Text Embeddings | p.2 | gs > 2.5 Real vs Hypothetical Embeddings
    Contextual retrieval techniques, such as Anthropic’s approach to augmenting chunks with additional information before embedding, have emerged as promising ways to reduce retrieval errors. Similarly, Hypothetical Document Embeddings (HyDE) [7] involve...
[2] - | AutoRAG: Automated Framework for optimization of Retrieval A | p.3 | iques > 2.1 Query Expansion > 2.1.2 HyDE
    The Hypothetical Document Embedding (HyDE) (Gao et al. (2022)) technique improves document retrieval by utilizing LLMs to generate a hypothetical passage to a query. This approach increases the embedding vector similarity that the hypothetical passag...
[3] - | QuOTE: Question-Oriented Text Embeddings | p.7 | xt Embeddings > 5.4 Comparison with HyDE
    A popular technique for query enrichment is **HyDE** , which generates a hypothetical document at _query time_ before embedding it and retrieving rele

# Generation

## Prompt Engineering

In [ ]:
NO_ANSWER = "I could not find the answer in the provided documents"

SYSTEM_PROMPT=f"""You are a research assistant that answers question about scientific papers.
Rules:
1. Answer ONLY using the numbered sources provided. Do not use outside knowledge.
2. After every claim, cite the supporting source number like [1] or [2][3].
3. If the sources do not contain the answer, reply exactly: "{NO_ANSWER}".
4. Be concise and precise. keep numbers and names excatly as they appear in sources."""

def build_context(R, results, max_chars_per_source=3000):
  blocks, sources =[], []
  for n, r in enumerate(results, 1):
    row = R.rows[r['idx']]
    text = r["text"][:max_chars_per_source]
    header = f"[{n}] {r['title']}, p.{r['page']}"
    header +=f", section: {r['section']}" if r["section"] else ")"
    blocks.append(f"{header}\n{text}")
    sources.append({"n": n, "title": r["title"], "arxiv_id": row["arxiv_id"],
                        "page": r["page"], "section": r["section"],
                        "chunk_id": r["chunk_id"]})
  return "\n\n".join(blocks), sources

def build_messages(question, context):
  user = f"Sources:\n{context}\n\nQuestion: {question}\n\nAnswer:"
  return [{"role":"system","content":SYSTEM_PROMPT},
          {"role":"user","content":user}]


## LLM Backend

In [ ]:
!pip install  groq

In [ ]:
from groq import Groq

API_FLAG = True
API_KEY = ""
GROQ_MODEL = "qwen/qwen3.8-27b"

GRQclient = Groq(
    api_key = API_KEY
)



In [ ]:
def llm_chat(messages,max_new_tokens=400):

  if API_FLAG:
    out = completion = GRQclient.chat.completions.create(
    model=GROQ_MODEL,
    messages=messages,
    temperature=0.6,
    max_completion_tokens=max_new_tokens,
    top_p=0.95,
    reasoning_effort="default",
    stream=True,
    stop=None
)
    return "".join(str(chunk.choices[0].delta.content) for chunk in out)

  text = _llm_tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
  inputs = _llm_tok(text, return_tensors="pt").to(_llm.device)
  with torch.no_grad():
    out = _llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
  return _llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:

def answer(R, question, *, top_k=5, max_new_tokens=400, **retrieve_kw):
    t0 = time.time()
    results, t_ret = retrieve(R, question, top_k=top_k, **retrieve_kw)
    context, sources = build_context(R, results)
    t1 = time.time()
    text = llm_chat(build_messages(question, context), max_new_tokens)
    t_gen = time.time() - t1

    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", text)})
    valid   = [n for n in cited if 1 <= n <= len(sources)]
    invalid = [n for n in cited if n not in valid]
    refused = NO_ANSWER.lower() in text.lower()

    return {
        "question": question,
        "answer": text,
        "refused": refused,
        "citations": [s for s in sources if s["n"] in valid],   # المصادر اللي اتذكرت فعلاً
        "invalid_citations": invalid,                           # أرقام الموديل اخترعها
        "uncited": (not refused) and not valid,                 # جاوب من غير أي citation
        "retrieved": sources,
        "latency": {"retrieval": round(t_ret, 2), "generation": round(t_gen, 2),
                    "total": round(time.time() - t0, 2)},
    }


def show_answer(a):
    print("Q:", a["question"])
    print("\nA:", a["answer"])
    print(f"\nrefused={a['refused']} | uncited={a['uncited']} | invalid={a['invalid_citations']}")
    print("Cited sources:")
    for s in a["citations"]:
        print(f"  [{s['n']}] {s['title'][:70]} | arXiv:{s['arxiv_id']} | p.{s['page']}")
    print("latency:", a["latency"])

In [ ]:
# 1) سؤال إجابته موجودة
show_answer(answer(R, "How does a cross-encoder reranker improve retrieval quality?"))

# 2) سؤال بمصطلح دقيق
show_answer(answer(R, "What is HyDE and how does it generate hypothetical documents?"))

# 3) سؤال مالوش إجابة في الداتا (لازم يرفض)
show_answer(answer(R, "What is the recipe for a chocolate cake?"))

# 4) بنفس الإعدادات لكن مع parent expansion (سياق أكبر)
show_answer(answer(R, "How does a cross-encoder reranker improve retrieval quality?",
                   expand_parents=True, top_k=3))

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Q: How does a cross-encoder reranker improve retrieval quality?

A: The cross-encoder re-ranking step improves retrieval quality by enhancing performance metrics for both sparse (BM25) and dense (Nomic) retrieval methods [1]. Specifically, adding this re-ranking step resulted in MRR increasing to 0.803 and MAP increasing to 0.499 [1]. For BM25, the re-ranking led to Recall@10 increasing to 0.635, MRR increasing to 0.886, and MAP increasing to 0.572 [1]. These results demonstrate that the re-ranking stage effectively enhances the performance of both retrieval methods [1].None

refused=False | uncited=False | invalid=[]
Cited sources:
  [1] Grounding Arabic LLMs in the Doha Historical Dictionary: Retrieval-Aug | arXiv:2603.23972 | p.13
latency: {'retrieval': 42.27, 'generation': 0.52, 'total': 42.8}
Q: What is HyDE and how does it generate hypothetical documents?

A: HyDE, or Hypothetical Document Embeddings, involves generating synthetic text based on the query and embedding it alongsid